# 📊 Paper 170: Multi-Class Trading Strategy Prediction

## 🎯 Research Goal
Predict which trading strategy (Momentum, Reversal, or Do Nothing) will yield the highest returns for different time windows using multiple ML models.

## 🔍 Methodology
- **Feature Windows:** 5, 10, 15, 20, 25, 30 days
- **Target Windows:** 5, 10, 15, 20, 25, 30 days  
- **Models:** RandomForest, GradientBoosting, SVM, LSTM
- **Classes:** 0=Momentum, 1=Reversal, 2=Do Nothing
- **Total Models:** 6 feature windows × 4 models × 6 target windows = **144 models**

## 📋 Structure
1. Setup & Data Loading
2. Multi-Class Target Calculation
3. Feature Creation
4. Model Training (with persistence)
5. Backtesting
6. Results Analysis

---

In [1]:
# ============================================================================
# Optimized Training Pipeline with Feature Caching
# ============================================================================

# Import required modules
import os
import json
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.kernel_approximation import RBFSampler
import xgboost as xgb
from tqdm import tqdm

# Directory configuration (ensure it's defined)
if 'DATA_DIR' not in globals():
    DATA_DIR = 'data/research'
    os.makedirs(DATA_DIR, exist_ok=True)

# Feature cache directory
FEATURES_CACHE_DIR = os.path.join(DATA_DIR, 'features_cache')
os.makedirs(FEATURES_CACHE_DIR, exist_ok=True)

print(f"💾 Feature cache directory: {FEATURES_CACHE_DIR}")

# ============================================================================
# Cache Management Functions
# ============================================================================

def save_features_to_cache(feature_df, window):
    """Save pre-calculated features to disk"""
    cache_file = os.path.join(FEATURES_CACHE_DIR, f'features_{window}d.pkl')
    print(f"   💾 Saving features to cache: {cache_file}")
    feature_df.to_pickle(cache_file)
    print(f"   ✅ Cached {len(feature_df.columns)} columns, {len(feature_df):,} records")

def load_features_from_cache(window):
    """Load pre-calculated features from disk if available"""
    cache_file = os.path.join(FEATURES_CACHE_DIR, f'features_{window}d.pkl')
    if os.path.exists(cache_file):
        print(f"   ♻️  Loading cached features from: {cache_file}")
        feature_df = pd.read_pickle(cache_file)
        print(f"   ✅ Loaded {len(feature_df.columns)} columns, {len(feature_df):,} records")
        return feature_df
    return None

def save_target_to_cache(target_series, window):
    """Save pre-calculated target to disk"""
    cache_file = os.path.join(FEATURES_CACHE_DIR, f'target_{window}d.pkl')
    print(f"   💾 Saving target to cache: {cache_file}")
    target_series.to_pickle(cache_file)

def load_target_from_cache(window):
    """Load pre-calculated target from disk if available"""
    cache_file = os.path.join(FEATURES_CACHE_DIR, f'target_{window}d.pkl')
    if os.path.exists(cache_file):
        print(f"   ♻️  Loading cached target from: {cache_file}")
        target_series = pd.read_pickle(cache_file)
        print(f"   ✅ Loaded target with {len(target_series):,} samples")
        return target_series
    return None

def clear_feature_cache():
    """Clear all cached features and targets"""
    import shutil
    if os.path.exists(FEATURES_CACHE_DIR):
        shutil.rmtree(FEATURES_CACHE_DIR)
        os.makedirs(FEATURES_CACHE_DIR)
        print("🗑️  Feature cache cleared!")
    else:
        print("ℹ️  No cache to clear")

print("✅ Loaded cache management functions")

💾 Feature cache directory: data/research/features_cache
✅ Loaded cache management functions


In [2]:
# ============================================================================
# Feature and Target Pre-calculation Functions
# ============================================================================

def precalculate_all_features(df, force_recalculate=False):
    """
    Pre-calculate features for all windows (done once!)
    If cached features exist, load them instead of recalculating
    """
    print("\n" + "="*70)
    print("🔧 PRE-CALCULATING FEATURES FOR ALL WINDOWS")
    print("="*70)
    
    if force_recalculate:
        print("⚠️  Force recalculate enabled - ignoring cache")
    
    feature_dfs = {}
    
    for window in FEATURE_WINDOWS:
        print(f"\n📊 Window {window} days:")
        
        # Try to load from cache first
        if not force_recalculate:
            cached_features = load_features_from_cache(window)
            if cached_features is not None:
                feature_dfs[window] = cached_features
                continue
        
        # Calculate features if not cached
        print(f"   🔧 Calculating features for {window}-day window...")
        feature_df = create_features_for_window(df, window)
        feature_dfs[window] = feature_df
        
        # Save to cache for future runs
        save_features_to_cache(feature_df, window)
    
    print("\n" + "="*70)
    print(f"✅ FEATURES READY FOR {len(FEATURE_WINDOWS)} WINDOWS")
    print("="*70)
    
    return feature_dfs

def precalculate_all_targets(df, force_recalculate=False):
    """
    Pre-calculate targets for all windows (done once!)
    If cached targets exist, load them instead of recalculating
    """
    print("\n" + "="*70)
    print("🎯 PRE-CALCULATING TARGETS FOR ALL WINDOWS")
    print("="*70)
    
    if force_recalculate:
        print("⚠️  Force recalculate enabled - ignoring cache")
    
    target_series = {}
    
    for window in TARGET_WINDOWS:
        print(f"\n🎯 Window {window} days:")
        
        # Try to load from cache first
        if not force_recalculate:
            cached_target = load_target_from_cache(window)
            if cached_target is not None:
                target_series[window] = cached_target
                
                # Show distribution
                counts = cached_target.value_counts().sort_index()
                print(f"   📊 Distribution:")
                print(f"      Class 0 (Momentum): {counts.get(0, 0):,} samples")
                print(f"      Class 1 (Reversal): {counts.get(1, 0):,} samples")
                print(f"      Class 2 (Do Nothing): {counts.get(2, 0):,} samples")
                continue
        
        # Calculate target if not cached
        print(f"   🔧 Calculating target for {window}-day window...")
        target = calculate_multiclass_target(df, window)
        target_series[window] = target
        
        # Show distribution
        counts = target.value_counts().sort_index()
        print(f"   📊 Distribution:")
        print(f"      Class 0 (Momentum): {counts.get(0, 0):,} samples")
        print(f"      Class 1 (Reversal): {counts.get(1, 0):,} samples")
        print(f"      Class 2 (Do Nothing): {counts.get(2, 0):,} samples")
        
        # Save to cache for future runs
        save_target_to_cache(target, window)
    
    print("\n" + "="*70)
    print(f"✅ TARGETS READY FOR {len(TARGET_WINDOWS)} WINDOWS")
    print("="*70)
    
    return target_series

print("✅ Loaded precalculate_all_features() and precalculate_all_targets()")


✅ Loaded precalculate_all_features() and precalculate_all_targets()


In [3]:


def train_lstm_model(X_train_scaled, X_test_scaled, y_train, y_test, model_name):
    """Train LSTM model"""
    try:
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense, Dropout
        from sklearn.metrics import accuracy_score, classification_report
        
        # Reshape for LSTM (samples, timesteps, features)
        # Use 5 timesteps for 5-day lookback
        timesteps = 5
        n_features = X_train_scaled.shape[1]
        
        # Pad or truncate to have consistent shape
        n_samples_train = (len(X_train_scaled) // timesteps) * timesteps
        n_samples_test = (len(X_test_scaled) // timesteps) * timesteps
        
        # FIX: Also truncate y_train and y_test to match
        X_train_reshaped = X_train_scaled[:n_samples_train].reshape(-1, timesteps, n_features)
        X_test_reshaped = X_test_scaled[:n_samples_test].reshape(-1, timesteps, n_features)
        
        # Reshape y to match - take only the last value of each sequence
        y_train_sequences = y_train[:n_samples_train].values if hasattr(y_train, 'values') else y_train[:n_samples_train]
        y_test_sequences = y_test[:n_samples_test].values if hasattr(y_test, 'values') else y_test[:n_samples_test]
        
        y_train_reshaped = y_train_sequences.reshape(-1, timesteps)[:, -1]  # Take last timestep
        y_test_reshaped = y_test_sequences.reshape(-1, timesteps)[:, -1]
        
        # Build LSTM model
        model = Sequential([
            LSTM(32, return_sequences=True, input_shape=(timesteps, n_features)),
            Dropout(0.2),
            LSTM(32, return_sequences=False),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(3, activation='softmax')  # 3 classes
        ])
        
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        # Train with early stopping
        early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        
        history = model.fit(
            X_train_reshaped, y_train_reshaped,
            epochs=20,
            batch_size=32,
            validation_split=0.2,
            callbacks=[early_stop],
            verbose=0
        )
        
        # Predict
        y_pred_proba = model.predict(X_test_reshaped, verbose=0)
        y_pred = y_pred_proba.argmax(axis=1)
        
        # Calculate metrics
        accuracy = accuracy_score(y_test_reshaped, y_pred)
        report = classification_report(y_test_reshaped, y_pred, output_dict=True, zero_division=0)
        
        metrics = {
            'accuracy': accuracy,
            'f1_weighted': report['weighted avg']['f1-score'],
            'precision_weighted': report['weighted avg']['precision'],
            'recall_weighted': report['weighted avg']['recall']
        }
        
        return model, metrics
        
    except Exception as e:
        print(f"LSTM training failed: {e}")
        import traceback
        traceback.print_exc()
        # Return a dummy model and metrics
        return None, {'accuracy': 0.0, 'f1_weighted': 0.0, 'precision_weighted': 0.0, 'recall_weighted': 0.0}


def train_single_model_optimized(X, y, model_name, feature_window, target_window):
    """
    Train a single model with TIME-BASED split (not random shuffle)
    
    CRITICAL: For financial time series, we must split chronologically:
    - Train on EARLY data (e.g., 2015-2022)
    - Test on LATE data (e.g., 2023-2024)
    
    This prevents data leakage and simulates real trading conditions.
    """
    
    # ============================================================
    # STEP 1: TIME-BASED SPLIT (Chronological)
    # ============================================================
    
    # Ensure data is sorted by index (should already be chronological)
    # Note: X and y should have the same index from the combined dataframe
    
    # Calculate split point (60% train, 40% test)
    import numpy as np
    split_idx = int(len(X) * 0.6)
    
    # Split chronologically - FIRST 60% for training, LAST 40% for testing
    X_train = X.iloc[:split_idx].copy()
    y_train = y.iloc[:split_idx].copy()
    X_test = X.iloc[split_idx:].copy()
    y_test = y.iloc[split_idx:].copy()
    
    print(f"     🔍 {model_name} - Time-based split:")
    print(f"        Train: {len(X_train):,} samples (earliest {len(X_train)/len(X)*100:.1f}%)")
    print(f"        Test:  {len(X_test):,} samples (latest {len(X_test)/len(X)*100:.1f}%)")
    
    # Show class distribution to ensure balance
    train_dist = y_train.value_counts().sort_index()
    test_dist = y_test.value_counts().sort_index()
    print(f"        Train classes: {dict(train_dist)}")
    print(f"        Test classes:  {dict(test_dist)}")
    
    # ============================================================
    # STEP 2: 5-LAYER DATA CLEANING
    # ============================================================
    
    # Layer 1: Remove all-NaN columns
    all_nan_cols = X_train.columns[X_train.isna().all()].tolist()
    if all_nan_cols:
        print(f"        🧹 Removing {len(all_nan_cols)} all-NaN columns")
        X_train = X_train.drop(columns=all_nan_cols)
        X_test = X_test.drop(columns=all_nan_cols)
    
    # Layer 2: Remove high-NaN columns (>80%)
    high_nan_cols = X_train.columns[X_train.isna().mean() > 0.8].tolist()
    if high_nan_cols:
        print(f"        🧹 Removing {len(high_nan_cols)} high-NaN columns (>80%)")
        X_train = X_train.drop(columns=high_nan_cols)
        X_test = X_test.drop(columns=high_nan_cols)
    
    # Layer 3: Ensure all columns are numeric
    numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
    if len(numeric_cols) < len(X_train.columns):
        print(f"        🧹 Keeping only {len(numeric_cols)} numeric columns")
        X_train = X_train[numeric_cols]
        X_test = X_test[numeric_cols]
    
    # Layer 4: Smart median filling
    for col in X_train.columns:
        if X_train[col].isna().any():
            median_val = X_train[col].median()
            if pd.isna(median_val):
                median_val = 0.0
            X_train[col].fillna(median_val, inplace=True)
            X_test[col].fillna(median_val, inplace=True)
    
    # Layer 5: Outlier clipping (1st to 99th percentile)
    for col in X_train.columns:
        if X_train[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
            q01 = X_train[col].quantile(0.01)
            q99 = X_train[col].quantile(0.99)
            X_train[col] = X_train[col].clip(q01, q99)
            X_test[col] = X_test[col].clip(q01, q99)
    
    print(f"        ✅ Clean data: {X_train.shape[1]} features")
    
    # ============================================================
    # STEP 3: TRAIN MODEL (with SVM memory optimization)
    # ============================================================
    
    if model_name == 'RandomForest':
        from sklearn.ensemble import RandomForestClassifier
        model = RandomForestClassifier(
            n_estimators=50,
            max_depth=10,
            min_samples_split=20,
            min_samples_leaf=10,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    
    elif model_name == 'GradientBoosting':
        from sklearn.ensemble import GradientBoostingClassifier
        model = GradientBoostingClassifier(
            n_estimators=50,
            max_depth=5,
            min_samples_split=20,
            min_samples_leaf=10,
            learning_rate=0.1,
            subsample=0.8,
            random_state=42
        )
    
    elif model_name == 'XGBoost':
        import xgboost as xgb
        model = xgb.XGBClassifier(
            n_estimators=150,
            max_depth=5,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            n_jobs=-1,
            eval_metric='mlogloss'
        )
    
    elif model_name == 'SVM':
        # ⭐ SVM MEMORY OPTIMIZATION
        from sklearn.kernel_approximation import RBFSampler
        from sklearn.linear_model import SGDClassifier
        from sklearn.pipeline import Pipeline
        
        # Limit SVM to last 200k samples (most recent data)
        max_svm_samples = 200000
        
        if len(X_train) > max_svm_samples:
            # Take LAST 200k samples (most recent chronological data)
            X_train_svm = X_train.iloc[-max_svm_samples:]
            y_train_svm = y_train.iloc[-max_svm_samples:]
            print(f"        ℹ️  SVM: Using last {len(X_train_svm):,} of {len(X_train):,} samples (most recent)")
        else:
            X_train_svm = X_train
            y_train_svm = y_train
        
        model = Pipeline([
            ('rbf_features', RBFSampler(
                gamma=0.1,
                n_components=400,      # Reduced from 1000 for memory efficiency
                random_state=42
            )),
            ('sgd_classifier', SGDClassifier(
                loss='log_loss',
                alpha=0.00001,
                max_iter=1500,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                class_weight='balanced',
                random_state=42,
                n_jobs=-1
            ))
        ])
        
        # Train on sampled data
        model.fit(X_train_svm, y_train_svm)
        
    elif model_name == 'LSTM':
        # LSTM implementation following paper architecture (Figure 4)
        import numpy as np
        from tensorflow import keras
        import tensorflow as tf
        
        # Set random seeds for reproducibility
        np.random.seed(42)
        tf.random.set_seed(42)
        
        # LSTM hyperparameters (from paper)
        n_timesteps = 5  # As shown in paper's Figure 4
        n_classes = 3    # Buy, Sell, Hold
        
        print(f"        🔄 Reshaping data for LSTM (timesteps={n_timesteps})...")
        
        # Helper function to reshape data for LSTM
        def reshape_for_lstm(X, n_timesteps):
            """Reshape 2D data into 3D sequences for LSTM"""
            X_values = X.values if hasattr(X, 'values') else X
            n_samples, n_features = X_values.shape
            
            X_sequences = []
            indices_kept = []
            
            for i in range(n_timesteps - 1, n_samples):
                sequence = X_values[i - n_timesteps + 1:i + 1, :]
                X_sequences.append(sequence)
                indices_kept.append(i)
            
            return np.array(X_sequences), indices_kept
        
        # Reshape training data into sequences
        X_train_lstm, train_indices = reshape_for_lstm(X_train, n_timesteps)
        y_train_lstm = y_train.iloc[train_indices].values
        
        # Reshape test data into sequences
        X_test_lstm, test_indices = reshape_for_lstm(X_test, n_timesteps)
        y_test_lstm = y_test.iloc[test_indices].values
        
        print(f"        📊 LSTM data: Train={X_train_lstm.shape[0]:,}, Test={X_test_lstm.shape[0]:,}")
        print(f"        📐 Shape: (samples, timesteps={n_timesteps}, features={X_train_lstm.shape[2]})")
        
        # Create LSTM model (Figure 4 architecture)
        n_features = X_train_lstm.shape[2]
        
        model = keras.Sequential([
            keras.layers.Input(shape=(n_timesteps, n_features)),
            keras.layers.LSTM(32, return_sequences=True, name='lstm_1'),
            keras.layers.LSTM(32, return_sequences=True, name='lstm_2'),
            keras.layers.LSTM(32, return_sequences=False, name='lstm_3'),
            keras.layers.Dense(n_classes, activation='softmax', name='dense_1')
        ])
        
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        
        print(f"        🏗️  LSTM Architecture: Input → LSTM(32) → LSTM(32) → LSTM(32) → Dense(3)")
        
        # Train model
        print(f"        🔧 Training LSTM...")
        history = model.fit(
            X_train_lstm, y_train_lstm,
            epochs=50,
            batch_size=32,
            validation_split=0.1,
            verbose=0,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor='val_loss',
                    patience=5,
                    restore_best_weights=True
                )
            ]
        )
        
        epochs_trained = len(history.history['loss'])
        final_train_acc = history.history['accuracy'][-1]
        final_val_acc = history.history['val_accuracy'][-1]
        
        print(f"        ✅ LSTM trained in {epochs_trained} epochs")
        print(f"           Train accuracy: {final_train_acc:.4f}, Val accuracy: {final_val_acc:.4f}")
        
        # Override test data with LSTM-formatted versions
        X_test = X_test_lstm
        y_test = y_test_lstm
    
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    # Train model (except SVM and LSTM which were already trained)
    if model_name not in ['SVM', 'LSTM']:
        model.fit(X_train, y_train)
    
    # ============================================================
    # STEP 4: EVALUATE MODEL
    # ============================================================
    
    from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
    
    # Make predictions
    y_pred = model.predict(X_test)
    # For LSTM/Keras models: convert probabilities to class labels
    if model_name == 'LSTM':
        # Keras returns probabilities (shape: n_samples, n_classes)
        # Convert to class labels using argmax
        y_pred = np.argmax(y_pred, axis=1)
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    # Per-class metrics
    CLASS_LABELS = {0: 'Buy', 1: 'Sell', 2: 'Hold'}
    report = classification_report(y_test, y_pred, 
                                   target_names=list(CLASS_LABELS.values()),
                                   output_dict=True,
                                   zero_division=0)
    
    metrics = {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro,
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'classification_report': report,
        'feature_window': feature_window,
        'target_window': target_window,
        'n_features': X_train.shape[1],
        'train_samples': len(X_train),
        'test_samples': len(X_test)
    }
    
    return model, metrics

    
print("✅ Loaded train_all_models_optimized() and train_single_model_optimized()")



✅ Loaded train_all_models_optimized() and train_single_model_optimized()


In [4]:
def train_all_models_optimized(df, force_recalculate_features=False):
    """
    Optimized training: Calculate features once, reuse for all models
    Features and targets are cached to disk for resuming training
    """
    # Ensure cache directory is available
    global FEATURES_CACHE_DIR
    if 'FEATURES_CACHE_DIR' not in globals():
        FEATURES_CACHE_DIR = os.path.join(DATA_DIR, 'features_cache')
        os.makedirs(FEATURES_CACHE_DIR, exist_ok=True)
    
    # Ensure other required directories are available
    global MODELS_DIR, RESULTS_DIR, LOG_FILE
    if 'MODELS_DIR' not in globals():
        MODELS_DIR = os.path.join(DATA_DIR, 'multiclass_models')
        os.makedirs(MODELS_DIR, exist_ok=True)
    if 'RESULTS_DIR' not in globals():
        RESULTS_DIR = os.path.join(DATA_DIR, 'multiclass_results')
        os.makedirs(RESULTS_DIR, exist_ok=True)
    if 'LOG_FILE' not in globals():
        LOG_FILE = os.path.join(DATA_DIR, 'training_log.json')
    
    print("\n" + "="*70)
    print("🚀 STARTING OPTIMIZED TRAINING PIPELINE")
    print("="*70)
    print(f"📊 Total models to train: {len(FEATURE_WINDOWS) * len(TARGET_WINDOWS) * len(MODEL_TYPES)}")
    print(f"   Feature windows: {FEATURE_WINDOWS}")
    print(f"   Target windows: {TARGET_WINDOWS}")
    print(f"   Model types: {MODEL_TYPES}")
    print(f"\n💾 Feature cache directory: {FEATURES_CACHE_DIR}")
    
    # Load training log
    log = load_training_log()
    
    # Step 1: Pre-calculate ALL features (once per window, with caching)
    print("\n" + "="*70)
    print("STEP 1: PRE-CALCULATING FEATURES (with caching)")
    print("="*70)
    feature_dfs = precalculate_all_features(df, force_recalculate=force_recalculate_features)
    
    # Step 2: Pre-calculate ALL targets (once per window, with caching)
    print("\n" + "="*70)
    print("STEP 2: PRE-CALCULATING TARGETS (with caching)")
    print("="*70)
    target_series = precalculate_all_targets(df, force_recalculate=force_recalculate_features)
    
    # Step 3: Train all model combinations (reusing pre-calculated features)
    print("\n" + "="*70)
    print("STEP 3: TRAINING ALL MODEL COMBINATIONS")
    print("="*70)
    
    total_models = len(FEATURE_WINDOWS) * len(TARGET_WINDOWS) * len(MODEL_TYPES)
    trained_count = 0
    skipped_count = 0
    
    for feature_window in FEATURE_WINDOWS:
        print(f"\n{'='*70}")
        print(f"📊 FEATURE WINDOW: {feature_window} days")
        print(f"{'='*70}")
        
        # Get pre-calculated features for this window
        feature_df = feature_dfs[feature_window]
        
        for target_window in TARGET_WINDOWS:
            print(f"\n  🎯 TARGET WINDOW: {target_window} days")
            
            # Get pre-calculated target for this window
            target = target_series[target_window]
            
            # Align features and target (same indices)
            # Align features and target (same indices)
            combined = feature_df.copy()
            combined['target'] = target
            combined = combined.dropna(subset=['target'])
            
            # ⭐ FILTER OUT NON-FEATURE COLUMNS (symbol, date, etc.)
            from training_feature_utils import NON_FEATURE_COLS
            
            exclude_cols = NON_FEATURE_COLS + ['target']
            valid_cols = [col for col in combined.columns if col not in exclude_cols]
            
            # Select only numeric columns to be extra safe
            X = combined[valid_cols].select_dtypes(include=['number'])
            y = combined['target']
            
            # Show what was excluded
            excluded_found = [col for col in combined.columns if col in NON_FEATURE_COLS]
            if excluded_found:
                print(f"     🔧 Excluded columns: {excluded_found}")
            
            print(f"     📊 Training data: {len(X):,} samples, {len(X.columns)} features")
            
            for model_name in MODEL_TYPES:
                model_key = f"f{feature_window}d_t{target_window}d_{model_name}"
                
                # Check if already trained
                if model_key in log:
                    acc = log[model_key].get('accuracy', 'N/A')
                    acc_str = f"{acc:.3f}" if isinstance(acc, (int, float)) else acc
                    print(f"     ⏭️  {model_name}: Already trained (accuracy: {acc_str})")
                    skipped_count += 1
                    continue
                
                print(f"\n     🔧 Training {model_name}...")
                
                try:
                    # Train the model
                    model, metrics = train_single_model_optimized(X, y, model_name, feature_window, target_window)
                    
                    # Save model
                    model_path = os.path.join(MODELS_DIR, f"{model_key}.pkl")
                    with open(model_path, 'wb') as f:
                        pickle.dump(model, f)
                    
                    # Save metrics
                    metrics_path = os.path.join(RESULTS_DIR, f"{model_key}_metrics.json")
                    with open(metrics_path, 'w') as f:
                        json.dump(metrics, f, indent=2)
                    
                    # Update log
                    log[model_key] = {
                        'feature_window': feature_window,
                        'target_window': target_window,
                        'model_type': model_name,
                        'accuracy': metrics.get('accuracy', 0),
                        'trained_at': pd.Timestamp.now().isoformat(),
                        'model_path': model_path,
                        'metrics_path': metrics_path
                    }
                    save_training_log(log)
                    
                    trained_count += 1
                    acc = metrics.get('accuracy', 0)
                    f1 = metrics.get('f1_weighted', metrics.get('f1', 0))
                    print(f"     ✅ {model_name}: accuracy={acc:.3f}, f1={f1:.3f}")
                    
                except Exception as e:
                    print(f"     ❌ {model_name} failed: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    continue
            
            progress = (trained_count + skipped_count) / total_models * 100
            print(f"\n  📊 Progress: {trained_count + skipped_count}/{total_models} ({progress:.1f}%)")
            print(f"     ✅ Trained: {trained_count}, ⏭️  Skipped: {skipped_count}")
    
    print("\n" + "="*70)
    print("🎉 TRAINING COMPLETE!")
    print("="*70)
    print(f"✅ Trained: {trained_count} models")
    print(f"⏭️  Skipped: {skipped_count} models (already trained)")
    print(f"📊 Total: {trained_count + skipped_count}/{total_models} models")
    print(f"\n💾 Feature cache saved to: {FEATURES_CACHE_DIR}")
    print(f"   (Will be reused on next run if data hasn't changed)")
    
    return log

print("✅ Loaded train_all_models_optimized()")

✅ Loaded train_all_models_optimized()


## ⚠️ IMPORTANT: How to Run This Notebook

### 🔄 After Restarting Kernel:

**You MUST run cells in order (Steps 1-8) before training:**

1. ✅ **Step 1**: Setup & Dependencies
2. ✅ **Step 2**: Configuration  
3. ✅ **Step 3**: Data Loading
4. ✅ **Step 4**: Multi-Class Target Calculation
5. ✅ **Step 5**: Feature Creation
6. ✅ **Step 6**: Training Log Management
7. ✅ **Step 7**: Model Training Functions
8. ✅ **Step 8**: Main Training Loop
9. 🚀 **Step 9**: Train All Models (execution)

### 📝 Quick Actions:

- **First time**: Click on a training cell → "Cell" → "Run All Above"
- **After error**: Kernel → "Restart & Run All"
- **Resume training**: Just run Step 9 (already trained models will be skipped)

### 🎯 Training Progress:

- View progress: Run Step 10 (View Training Progress)
- Check models: Run Step 12 (Performance Analysis)
- Training persists across restarts (uses `training_log.json`)

---

## Step 1: Setup & Dependencies

In [5]:
import numpy as np
print(f"   🔬 NumPy: {np.__version__}")

   🔬 NumPy: 1.26.4


In [6]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Deep Learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, InputLayer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Utilities
import os
import json
import pickle
import gc
from tqdm import tqdm

print("✅ All dependencies loaded successfully!")
print(f"   📊 Pandas: {pd.__version__}")
print(f"   🤖 TensorFlow: {tf.__version__}")
print(f"   🔬 NumPy: {np.__version__}")

✅ All dependencies loaded successfully!
   📊 Pandas: 2.3.3
   🤖 TensorFlow: 2.20.0
   🔬 NumPy: 1.26.4


## Step 2: Configuration & Global Settings

## Step 3: Data Loading

In [7]:
def load_research_data():
    """Load saved data from Phase 1"""
    print("📂 Loading saved research data...")
    
    # Try to load from pickle first (faster)
    pkl_file = f'{DATA_DIR}/sp500_stock_data_latest.pkl'
    csv_file = f'{DATA_DIR}/sp500_stock_data_latest.csv'
    
    if os.path.exists(pkl_file):
        print(f"   📦 Loading from pickle: {pkl_file}")
        df = pd.read_pickle(pkl_file)
    elif os.path.exists(csv_file):
        print(f"   📄 Loading from CSV: {csv_file}")
        df = pd.read_csv(csv_file)
        df['date'] = pd.to_datetime(df['date'])
    else:
        print("   ❌ No saved data found!")
        print("   Please run Phase 1 data collection first")
        return None, None
    
    # Load metadata
    metadata_file = f'{DATA_DIR}/sp500_metadata_latest.json'
    if os.path.exists(metadata_file):
        with open(metadata_file, 'r') as f:
            metadata = json.load(f)
    else:
        metadata = {}
    
    print("✅ Data loaded successfully!")
    print(f"   📊 Records: {len(df):,}")
    print(f"   🏢 Symbols: {df['symbol'].nunique()}")
    print(f"   📅 Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"   📋 Columns: {len(df.columns)}")
    
    return df, metadata

# Load data
stock_data, metadata = load_research_data()

if stock_data is not None:
    print("\n🎯 Ready to proceed with multi-class training!")
else:
    print("\n❌ Cannot proceed without data")

📂 Loading saved research data...
   📦 Loading from pickle: data/research/sp500_stock_data_latest.pkl
✅ Data loaded successfully!
   📊 Records: 669,825
   🏢 Symbols: 285
   📅 Date range: 2015-01-02 00:00:00 to 2024-12-30 00:00:00
   📋 Columns: 33

🎯 Ready to proceed with multi-class training!


In [8]:
# Windows configuration
FEATURE_WINDOWS = [5, 10, 15, 20, 25, 30]
TARGET_WINDOWS = [5, 10, 15, 20, 25, 30]

# Model configuration
MODEL_TYPES = ['RandomForest', 'GradientBoosting', 'XGBoost', 'SVM', 'LSTM']

# Class labels
CLASS_LABELS = {
    0: 'Momentum',
    1: 'Reversal',
    2: 'Do Nothing'
}

# Directory configuration
DATA_DIR = 'data/research'
MODEL_DIR = f'{DATA_DIR}/multiclass_models'
RESULTS_DIR = f'{DATA_DIR}/multiclass_results'
LOG_FILE = f'{DATA_DIR}/training_log.json'

# Create directories
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Training configuration
TEST_SIZE = 0.4
RANDOM_STATE = 42
TOP_PERCENT = 0.2  # Top 20% for momentum/reversal stocks

print("✅ Configuration complete!")
print(f"   📁 Model directory: {MODEL_DIR}")
print(f"   📁 Results directory: {RESULTS_DIR}")
print(f"   📊 Feature windows: {FEATURE_WINDOWS}")
print(f"   🎯 Target windows: {TARGET_WINDOWS}")
print(f"   🤖 Models: {MODEL_TYPES}")
print(f"   🔢 Total models to train: {len(FEATURE_WINDOWS) * len(TARGET_WINDOWS) * len(MODEL_TYPES)}")

✅ Configuration complete!
   📁 Model directory: data/research/multiclass_models
   📁 Results directory: data/research/multiclass_results
   📊 Feature windows: [5, 10, 15, 20, 25, 30]
   🎯 Target windows: [5, 10, 15, 20, 25, 30]
   🤖 Models: ['RandomForest', 'GradientBoosting', 'XGBoost', 'SVM', 'LSTM']
   🔢 Total models to train: 180


## Step 4: Multi-Class Target Calculation

For each time point and target window:
1. Calculate momentum yield (top 20% momentum stocks' future return)
2. Calculate reversal yield (top 20% reversal stocks' future return)
3. Choose best strategy: Momentum (0), Reversal (1), or Do Nothing (2)

In [9]:
def calculate_multiclass_target(df, target_window, top_percent=0.2):
    """
    Calculate multi-class target: Momentum (0), Reversal (1), or Do Nothing (2)
    
    Parameters:
    -----------
    df : DataFrame
        Stock data with price information
    target_window : int
        Target prediction window in days
    top_percent : float
        Percentage of top stocks to consider (default 0.2 for top 20%)
    
    Returns:
    --------
    target : Series
        Multi-class target (0=Momentum, 1=Reversal, 2=Do Nothing)
    """
    
    # Transaction costs: 0.03% commission + 0.1% tax + buffer ≈ 0.3%
    TRANSACTION_COST = 0.003  # 0.3% per round-trip trade
    
    print(f"    🎯 Calculating multi-class target for {target_window}-day window...")
    
    # Sort by symbol and date
    df = df.sort_values(['symbol', 'date']).copy()
    
    # Calculate momentum scores (past performance)
    momentum_scores = df.groupby('symbol')['close'].pct_change(target_window)
    
    # Calculate reversal scores (contrarian - opposite of momentum)
    reversal_scores = -momentum_scores
    
    # Calculate future returns (what actually happens)
    future_returns = df.groupby('symbol')['close'].pct_change(target_window).shift(-target_window)
    
    # Initialize target array
    target = pd.Series(2, index=df.index)  # Default: Do Nothing
    
    # For each date, calculate yields and assign target
    dates = df['date'].unique()
    print(f"    📅 Processing {len(dates):,} dates...")
    
    momentum_count = 0
    reversal_count = 0
    nothing_count = 0
    
    for date in tqdm(dates, desc="    Calculating targets"):
        date_mask = df['date'] == date
        date_indices = df[date_mask].index
        
        # Get stocks available on this date
        date_momentum = momentum_scores[date_mask].dropna()
        date_reversal = reversal_scores[date_mask].dropna()
        date_future_returns = future_returns[date_mask].dropna()
        
        if len(date_momentum) == 0 or len(date_future_returns) == 0:
            continue
        
        # Get top momentum stocks (highest past returns)
        momentum_threshold = date_momentum.quantile(1 - top_percent)
        top_momentum_stocks = date_momentum[date_momentum >= momentum_threshold].index
        
        # Calculate momentum yield (average future return of top momentum stocks)
        momentum_future_returns = date_future_returns[date_future_returns.index.isin(top_momentum_stocks)]
        momentum_yield = momentum_future_returns.mean() if len(momentum_future_returns) > 0 else 0
        
        # Get top reversal stocks (lowest past returns, expecting reversal)
        reversal_threshold = date_reversal.quantile(1 - top_percent)
        top_reversal_stocks = date_reversal[date_reversal >= reversal_threshold].index
        
        # Calculate reversal yield (average future return of top reversal stocks)
        reversal_future_returns = date_future_returns[date_future_returns.index.isin(top_reversal_stocks)]
        reversal_yield = reversal_future_returns.mean() if len(reversal_future_returns) > 0 else 0
        
        # Assign target class based on best strategy
        if momentum_yield > reversal_yield and momentum_yield > TRANSACTION_COST:
            target[date_indices] = 0  # Momentum
            momentum_count += len(date_indices)
        elif reversal_yield > momentum_yield and reversal_yield > TRANSACTION_COST:
            target[date_indices] = 1  # Reversal
            reversal_count += len(date_indices)
        else:
            target[date_indices] = 2  # Do Nothing
            nothing_count += len(date_indices)
    
    # Print distribution
    total = momentum_count + reversal_count + nothing_count
    if total > 0:
        print(f"\n    ✅ Target distribution for {target_window}-day window:")
        print(f"       🚀 Momentum: {momentum_count:,} ({momentum_count/total*100:.1f}%)")
        print(f"       🔄 Reversal: {reversal_count:,} ({reversal_count/total*100:.1f}%)")
        print(f"       ⏸️  Do Nothing: {nothing_count:,} ({nothing_count/total*100:.1f}%)")
    
    return target

# Test the function
if stock_data is not None:
    print("\n🧪 Testing multi-class target calculation...")
    test_target = calculate_multiclass_target(stock_data.head(1000), target_window=5)
    print(f"\n✅ Target calculation working correctly!")


🧪 Testing multi-class target calculation...
    🎯 Calculating multi-class target for 5-day window...
    📅 Processing 1,000 dates...


    Calculating targets: 100%|██████████| 1000/1000 [00:01<00:00, 837.31it/s]


    ✅ Target distribution for 5-day window:
       🚀 Momentum: 0 (0.0%)
       🔄 Reversal: 0 (0.0%)
       ⏸️  Do Nothing: 990 (100.0%)

✅ Target calculation working correctly!


## Step 5: Feature Creation Functions

### 📊 Comprehensive Feature Calculation Functions

**These functions create all 80+ features from the paper:**

1. **calculate_momentum_features()** - ~25 momentum features
   - Price momentum (multiple windows)
   - Cross-sectional ranks
   - Persistence, acceleration, relative strength

2. **calculate_reversal_features()** - ~20 reversal features
   - Mean reversion
   - Volatility mean reversion
   - RSI-based reversal

3. **calculate_volume_features()** - ~12 volume features
   - Volume momentum
   - Volume-price relationships
   - Abnormal volume

**Note:** These create features for ALL windows, then we filter to the specific window in `create_features_for_window()`.

## Step 3: Technical Indicator Functions


In [10]:
# Technical Indicator Functions (from paper)

def calculate_rsi_custom(prices, window=14):
    """Calculate RSI without talib dependency"""
    if isinstance(prices, np.ndarray):
        prices = pd.Series(prices)
    
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.values

def calculate_macd_custom(prices, fast=12, slow=26, signal=9):
    """Calculate MACD without talib dependency"""
    if isinstance(prices, np.ndarray):
        prices = pd.Series(prices)
    
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    macd_signal = macd_line.ewm(span=signal, adjust=False).mean()
    macd_histogram = macd_line - macd_signal
    return macd_line.values, macd_signal.values, macd_histogram.values

def calculate_cci_custom(high, low, close, window=14):
    """Calculate CCI (Commodity Channel Index)"""
    if isinstance(high, np.ndarray):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
    
    tp = (high + low + close) / 3
    sma_tp = tp.rolling(window).mean()
    mad = tp.rolling(window).apply(lambda x: np.abs(x - x.mean()).mean())
    cci = (tp - sma_tp) / (0.015 * mad)
    return cci.values

def calculate_kdj_custom(high, low, close, window=9, smooth_k=3, smooth_d=3):
    """Calculate KDJ indicator"""
    if isinstance(high, np.ndarray):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
    
    low_min = low.rolling(window).min()
    high_max = high.rolling(window).max()
    rsv = (close - low_min) / (high_max - low_min) * 100
    k = rsv.ewm(com=smooth_k-1, adjust=False).mean()
    d = k.ewm(com=smooth_d-1, adjust=False).mean()
    return k.values, d.values

def calculate_williams_r_custom(high, low, close, window=14):
    """Calculate Williams %R"""
    if isinstance(high, np.ndarray):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
    
    highest_high = high.rolling(window).max()
    lowest_low = low.rolling(window).min()
    willr = -100 * (highest_high - close) / (highest_high - lowest_low)
    return willr.values

def calculate_parabolic_sar_custom(high, low, close, af=0.02, max_af=0.2):
    """Calculate Parabolic SAR"""
    if isinstance(high, np.ndarray):
        high = pd.Series(high)
        low = pd.Series(low)
        close = pd.Series(close)
    
    # Simplified SAR calculation
    sar = close.copy()
    for i in range(1, len(close)):
        if i == 0:
            sar.iloc[i] = low.iloc[i]
        else:
            sar.iloc[i] = sar.iloc[i-1] + af * (high.iloc[i-1] - sar.iloc[i-1])
    
    return sar.values

print("✅ Loaded technical indicator functions")


✅ Loaded technical indicator functions


In [11]:
# Feature Calculation Functions (Paper's Approach - Window-Specific)
import numpy as np
import pandas as pd

def calculate_momentum_features(df, window):
    """
    Calculate momentum features for a specific window (matches paper's approach)
    All features use the SAME window parameter
    
    Based on paper's Table 2: mom_amplitude, mom_roc, mom_current_rtn, etc.
    """
    features = {}
    
    print(f"    📈 Calculating momentum features for {window}-day window...")
    
    # 1. mom_current_rtn - current period return
    features['mom_current_rtn'] = df.groupby('symbol')['close'].pct_change(window)
    
    # 2. mom_current_rtn_std - std of returns
    features['mom_current_rtn_std'] = df.groupby('symbol')['close'].pct_change(1).rolling(window).std().reset_index(level=0, drop=True)
    
    # 3. mom_amplitude - price amplitude (high-low)/close
    amplitude = (df['high'] - df['low']) / df['close']
    features['mom_amplitude'] = amplitude
    features['mom_amplitude_std'] = amplitude.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 4. mom_roc - Rate of Change
    roc = df.groupby('symbol')['close'].transform(lambda x: (x - x.shift(window)) / x.shift(window) * 100)
    features['mom_roc'] = roc
    features['mom_roc_std'] = roc.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 5. mom_ps - momentum persistence (how consistent the direction is)
    returns = df.groupby('symbol')['close'].pct_change(1)
    features['mom_ps'] = returns.groupby(df['symbol']).transform(
        lambda x: x.rolling(window).apply(lambda y: (y > 0).sum() / len(y) if len(y) > 0 else 0.5)
    )
    mom_ps_series = features['mom_ps']
    features['mom_ps_std'] = mom_ps_series.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 6. mom_turnover - volume turnover
    turnover = df['volume'] * df['close']
    features['mom_turnover'] = turnover
    features['mom_turnover_std'] = turnover.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 7. mom_yield_dispersion - cross-sectional yield dispersion
    returns_cs = df.groupby('symbol')['close'].pct_change(window)
    yield_disp = returns_cs.groupby(df['date']).transform('std')
    features['mom_yield_dispersion'] = yield_disp
    features['mom_yield_dispersion_std'] = yield_disp.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 8. mom_lb - Lower Band (similar to Bollinger lower band concept)
    rolling_mean = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window).mean())
    rolling_std = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window).std())
    mom_lb = rolling_mean - 2 * rolling_std
    features['mom_lb'] = mom_lb
    features['mom_lb_std'] = mom_lb.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 9. Fundamental momentum (price relative to fundamentals)
    if 'market_cap' in df.columns:
        market_cap_ratio = df['close'] / (df['market_cap'] / 1e9)  # Price to market cap ratio
        features['mom_market_cap'] = market_cap_ratio
        features['mom_market_cap_std'] = market_cap_ratio.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    if 'cir_cap' in df.columns:
        cir_cap_ratio = df['close'] / (df['cir_cap'] / 1e9)
        features['mom_cir_cap'] = cir_cap_ratio
        features['mom_cir_cap_std'] = cir_cap_ratio.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # Add fundamental ratios with momentum trends
    for ratio in ['pe', 'pb', 'ps', 'pcf']:
        if ratio in df.columns:
            ratio_pct = df.groupby('symbol')[ratio].pct_change(window)
            features[f'mom_{ratio}'] = ratio_pct
            features[f'mom_{ratio}_std'] = df.groupby('symbol')[ratio].transform(lambda x: x.rolling(window).std())
    
    print(f"    ✅ Generated {len(features)} momentum features")
    return features

def calculate_reversal_features(df, window):
    """
    Calculate reversal features for a specific window (matches paper's approach)
    All features use the SAME window parameter
    
    Based on paper's Table 2: rev_amplitude, rev_roc, rev_current_rtn, etc.
    """
    features = {}
    
    print(f"    📉 Calculating reversal features for {window}-day window...")
    
    # 1. rev_current_rtn - reversal of current return (contrarian)
    features['rev_current_rtn'] = -df.groupby('symbol')['close'].pct_change(window)
    
    # 2. rev_current_rtn_std
    features['rev_current_rtn_std'] = df.groupby('symbol')['close'].pct_change(1).rolling(window).std().reset_index(level=0, drop=True)
    
    # 3. rev_amplitude - reversal amplitude
    amplitude = (df['high'] - df['low']) / df['close']
    features['rev_amplitude'] = amplitude
    features['rev_amplitude_std'] = amplitude.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 4. rev_roc - reversal ROC (opposite momentum)
    roc = -df.groupby('symbol')['close'].transform(lambda x: (x - x.shift(window)) / x.shift(window) * 100)
    features['rev_roc'] = roc
    features['rev_roc_std'] = roc.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 5. rev_turnover
    turnover = df['volume'] * df['close']
    features['rev_turnover'] = turnover
    features['rev_turnover_std'] = turnover.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 6. rev_yield_dispersion
    returns_cs = df.groupby('symbol')['close'].pct_change(window)
    yield_disp = returns_cs.groupby(df['date']).transform('std')
    features['rev_yield_dispersion'] = yield_disp
    features['rev_yield_dispersion_std'] = yield_disp.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 7. rev_lb - reversal lower band
    rolling_mean = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window).mean())
    rolling_std = df.groupby('symbol')['close'].transform(lambda x: x.rolling(window).std())
    rev_lb = rolling_mean - 2 * rolling_std
    features['rev_lb'] = rev_lb
    features['rev_lb_std'] = rev_lb.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 8. Fundamental reversals
    if 'market_cap' in df.columns:
        market_cap_pct = -df.groupby('symbol')['market_cap'].pct_change(window)
        features['rev_market_cap'] = market_cap_pct
        features['rev_market_cap_std'] = df.groupby('symbol')['market_cap'].transform(lambda x: x.rolling(window).std())
    
    if 'cir_cap' in df.columns:
        cir_cap_pct = -df.groupby('symbol')['cir_cap'].pct_change(window)
        features['rev_cir_cap'] = cir_cap_pct
        features['rev_cir_cap_std'] = cir_cap_pct.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # Add fundamental ratios with reversal trends
    for ratio in ['pe', 'pb', 'ps', 'pcf']:
        if ratio in df.columns:
            ratio_pct = -df.groupby('symbol')[ratio].pct_change(window)
            features[f'rev_{ratio}'] = ratio_pct
            features[f'rev_{ratio}_std'] = df.groupby('symbol')[ratio].transform(lambda x: x.rolling(window).std())
    
    print(f"    ✅ Generated {len(features)} reversal features")
    return features


def calculate_volume_features(df, window):
    """
    Calculate volume features for a specific window
    """
    features = {}
    
    print(f"    📊 Calculating volume features for {window}-day window...")
    
    # 1. Volume change (vol_change)
    features['vol_change'] = df.groupby('symbol')['volume'].pct_change(window)
    
    # 2. Volume turnover
    turnover = df['volume'] * df['close']
    features['turnover'] = turnover
    features['turnover_std'] = turnover.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    # 3. On-Balance Volume (OBV)
    price_change = df.groupby('symbol')['close'].diff()
    obv_change = df['volume'] * np.sign(price_change)
    features['obv'] = obv_change.groupby(df['symbol']).transform(lambda x: x.rolling(window).sum())
    
    # 4. Yield dispersion (already calculated in momentum, but keep for completeness)
    returns = df.groupby('symbol')['close'].pct_change(window)
    yield_disp = returns.groupby(df['date']).transform('std')
    features['yield_dispersion'] = yield_disp
    features['yield_dispersion_std'] = yield_disp.groupby(df['symbol']).transform(lambda x: x.rolling(window).std())
    
    print(f"    ✅ Generated {len(features)} volume features")
    return features

print("✅ Loaded all feature calculation functions (window-specific approach)")


✅ Loaded all feature calculation functions (window-specific approach)


In [12]:
def create_features_for_window(df, feature_window):
    """
    Create features based on a specific window size using paper's approach
    All ~80-100 features are calculated using the SAME window
    
    Parameters:
    -----------
    df : DataFrame
        Stock data
    feature_window : int
        Feature calculation window in days (5, 10, 15, 20, 25, 30)
    
    Returns:
    --------
    feature_df : DataFrame
        DataFrame with ~80-100 window-specific features
    """
    print(f"\n  📊 Creating comprehensive features for {feature_window}-day window...")
    
    # Start with basic columns
    feature_df = df[['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']].copy()
    
    # Add fundamental data if available
    fundamental_cols = ['pe', 'pb', 'ps', 'pcf', 'market_cap', 'cir_cap']
    for col in fundamental_cols:
        if col in df.columns:
            feature_df[col] = df[col]
    
    # Add sector (categorical)
    if 'sector' in df.columns:
        feature_df['sector'] = df['sector']
    
    # Calculate ALL features using the SAME window
    print(f"    🔧 Calculating momentum features...")
    momentum_feats = calculate_momentum_features(df, feature_window)
    
    print(f"    🔧 Calculating reversal features...")
    reversal_feats = calculate_reversal_features(df, feature_window)
    
    print(f"    🔧 Calculating volume features...")
    volume_feats = calculate_volume_features(df, feature_window)
    
    # Technical indicators (using window parameter)
    print(f"    🔧 Calculating technical indicators...")
    tech_feats = {}
    
    # RSI (use window as period)
    for symbol in df['symbol'].unique():
        mask = df['symbol'] == symbol
        symbol_data = df[mask]['close'].values
        if len(symbol_data) >= feature_window:
            rsi = calculate_rsi_custom(symbol_data, window=feature_window)
            tech_feats.setdefault('rsi', pd.Series(index=df.index, dtype=float))[mask] = rsi
    
    # MACD (scaled to window)
    fast = max(int(feature_window * 0.5), 3)
    slow = feature_window
    signal = max(int(feature_window * 0.3), 3)
    
    for symbol in df['symbol'].unique():
        mask = df['symbol'] == symbol
        symbol_data = df[mask]['close'].values
        if len(symbol_data) >= slow:
            macd_line, macd_signal, macd_hist = calculate_macd_custom(symbol_data, fast, slow, signal)
            tech_feats.setdefault('macd', pd.Series(index=df.index, dtype=float))[mask] = macd_line
            tech_feats.setdefault('macd_signal', pd.Series(index=df.index, dtype=float))[mask] = macd_signal
            tech_feats.setdefault('macd_histogram', pd.Series(index=df.index, dtype=float))[mask] = macd_hist
    
    # CCI (use window as period)
    tech_feats['cci'] = calculate_cci_custom(df['high'].values, df['low'].values, df['close'].values, window=feature_window)
    
    # KDJ (use window as period)
    kdj_k, kdj_d = calculate_kdj_custom(df['high'].values, df['low'].values, df['close'].values, window=feature_window)
    tech_feats['kdj_slow_k'] = kdj_k
    tech_feats['kdj_slow_d'] = kdj_d
    
    # Williams %R (use window as period)
    tech_feats['willr'] = calculate_williams_r_custom(df['high'].values, df['low'].values, df['close'].values, window=feature_window)
    
    # SAR (use window-based parameters)
    tech_feats['sar'] = calculate_parabolic_sar_custom(df['high'].values, df['low'].values, df['close'].values)
    
    # EMA and SMA (use window as period)
    tech_feats['ema'] = df.groupby('symbol')['close'].transform(lambda x: x.ewm(span=feature_window, adjust=False).mean())
    tech_feats['sma'] = df.groupby('symbol')['close'].transform(lambda x: x.rolling(feature_window).mean())
    
    # Combine all features
    all_features = {**momentum_feats, **reversal_feats, **volume_feats, **tech_feats}
    
    print(f"    📊 Total features calculated: {len(all_features)}")
    
    # Add to dataframe
    for feat_name, feat_values in all_features.items():
        if isinstance(feat_values, pd.Series):
            feature_df[feat_name] = feat_values
        else:
            feature_df[feat_name] = feat_values
    
    # Clean data
    feature_df = feature_df.replace([np.inf, -np.inf], np.nan)
    
    print(f"    ✅ Final features: {len(feature_df.columns)} columns")
    print(f"    📊 Dataset size: {len(feature_df):,} records")
    
    return feature_df

print("✅ Updated create_features_for_window() with paper's approach!")


✅ Updated create_features_for_window() with paper's approach!


## Step 6: Training Log Management (Persistence)

In [13]:
def load_training_log():
    """Load training log to track which models are already trained"""
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_training_log(log):
    """Save training log"""
    with open(LOG_FILE, 'w') as f:
        json.dump(log, f, indent=2)

def is_model_trained(feature_window, target_window, model_name):
    """Check if a specific model is already trained"""
    log = load_training_log()
    key = f"f{feature_window}d_t{target_window}d_{model_name}"
    return key in log and log[key].get('status') == 'completed'

def mark_model_trained(feature_window, target_window, model_name, metrics):
    """Mark a model as trained in the log"""
    log = load_training_log()
    key = f"f{feature_window}d_t{target_window}d_{model_name}"
    log[key] = {
        'status': 'completed',
        'feature_window': feature_window,
        'target_window': target_window,
        'model_name': model_name,
        'metrics': metrics,
        'timestamp': datetime.now().isoformat()
    }
    save_training_log(log)

def get_training_progress():
    """Get training progress statistics"""
    log = load_training_log()
    total_models = len(FEATURE_WINDOWS) * len(TARGET_WINDOWS) * len(MODEL_TYPES)
    trained_models = len([k for k, v in log.items() if v.get('status') == 'completed'])
    return trained_models, total_models

# Check current progress
trained, total = get_training_progress()
print(f"\n📊 Training Progress: {trained}/{total} models ({trained/total*100 if total > 0 else 0:.1f}%)")


📊 Training Progress: 0/180 models (0.0%)


## Step 7: Model Training Functions

In [14]:
def get_model_filename(feature_window, target_window, model_name):
    """Generate model filename"""
    return f"{MODEL_DIR}/f{feature_window}d_t{target_window}d_{model_name}"

def get_metrics_filename(feature_window, target_window, model_name):
    """Generate metrics filename"""
    return f"{RESULTS_DIR}/f{feature_window}d_t{target_window}d_{model_name}_metrics.json"

def train_sklearn_model(X_train, X_test, y_train, y_test, model_name):
    """
    Train a scikit-learn model
    
    Parameters:
    -----------
    X_train, X_test : array-like
        Training and test features
    y_train, y_test : array-like
        Training and test targets
    model_name : str
        Name of the model ('RandomForest', 'GradientBoosting', 'SVM')
    
    Returns:
    --------
    model : trained model
    metrics : dict with evaluation metrics
    """
    # Create model
    if model_name == 'RandomForest':
        model = RandomForestClassifier(
            n_estimators=50,           # Reduced from 100 (fewer trees = smaller model)
            max_depth=10,              # Limit tree depth to prevent overfitting
            min_samples_split=20,      # Require more samples to split (creates smaller trees)
            min_samples_leaf=10,       # Require more samples per leaf
            max_features='sqrt',       # Use sqrt of features (faster, smaller)
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    elif model_name == 'GradientBoosting':
        model = GradientBoostingClassifier(
            n_estimators=50,           # Reduced from 100
            max_depth=5,               # Limit tree depth
            min_samples_split=20,      # Require more samples to split
            min_samples_leaf=10,       # Require more samples per leaf
            random_state=RANDOM_STATE
        )
    elif model_name == 'XGBoost':
        # Optimized XGBoost: More trees + proper depth + light regularization
        model = xgb.XGBClassifier(
            n_estimators=150,      # More trees than original (was 50, tried 100)
            max_depth=5,           # Restore optimal depth for XGBoost (was 5 → 3 → 5)
            learning_rate=0.08,    # Slightly lower for stability with more trees
            subsample=0.9,         # Light row sampling (was 0.8 - too aggressive)
            colsample_bytree=0.9,  # Light column sampling (was 0.8 - too aggressive)
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric='mlogloss'  # Multi-class log loss
        )
    elif model_name == 'SVM':
        # IMPROVED RBF Kernel Approximation with better hyperparameters
        from sklearn.kernel_approximation import RBFSampler
        from sklearn.linear_model import SGDClassifier
        from sklearn.pipeline import Pipeline
        
        model = Pipeline([
            ('rbf_features', RBFSampler(
                gamma=0.1, 
                n_components=1000,      # Increased from 300 for better approximation
                random_state=RANDOM_STATE
            )),
            ('sgd_classifier', SGDClassifier(
                loss='log_loss',        # Use log loss for probability support
                alpha=0.00001,          # Reduced regularization (was default 0.0001)
                max_iter=2000,          # Increased for convergence (was 1000)
                early_stopping=True,    # Stop when validation doesn't improve
                validation_fraction=0.1, # 10% for early stopping
                n_iter_no_change=10,    # Stop if no improvement for 10 iterations
                class_weight='balanced', # Handle class imbalance
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ])
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Calculate metrics
    metrics = {
        'accuracy': float(accuracy_score(y_test, y_pred)),
        'precision_macro': float(precision_score(y_test, y_pred, average='macro', zero_division=0)),
        'recall_macro': float(recall_score(y_test, y_pred, average='macro', zero_division=0)),
        'f1_macro': float(f1_score(y_test, y_pred, average='macro', zero_division=0)),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'classification_report': classification_report(y_test, y_pred, target_names=list(CLASS_LABELS.values()), output_dict=True)
    }
    
    return model, metrics


### 📝 Note on SVM Implementation

**RBF Kernel Approximation for SVM:**

The paper uses SVM with RBF kernel, which has O(n²) complexity. With 670K samples, exact RBF SVM would take:
- **12-24 hours per model**
- **18-36 days for all 36 SVM models**
- **Similar to the paper's 4-year research timeline**

**Our Approach:**
We use **RBF Kernel Approximation** (Random Fourier Features) which:
- ✅ Approximates RBF kernel mathematically
- ✅ Maintains SVM properties
- ✅ **10-100x faster** (minutes instead of hours)
- ✅ **~95% accuracy** of exact RBF SVM
- ✅ Standard practice in ML research

**Reference:**
- Rahimi, A., & Recht, B. (2007). "Random Features for Large-Scale Kernel Machines". *NIPS*.
- This method is widely accepted when exact kernel methods are computationally prohibitive.

**Implementation:**
```python
Pipeline([
    ('rbf_features', RBFSampler(gamma=0.1, n_components=100)),
    ('sgd_classifier', SGDClassifier(loss='hinge'))
])
```

This gives us **SVM-like results in practical time** while maintaining research validity.

### 📊 Model Size Optimization

**Why optimized parameters?**

With 144 models to train, disk space is critical:

| Parameter | Original | Optimized | Impact |
|-----------|----------|-----------|--------|
| `n_estimators` | 100 | 50 | **50% fewer trees** |
| `max_depth` | None | 10 | **Prevents deep trees** |
| `min_samples_split` | 2 | 20 | **Smaller trees** |
| `min_samples_leaf` | 1 | 10 | **Fewer leaf nodes** |
| `max_features` | auto | sqrt | **Faster training** |

**Expected Results:**
- 📉 Model size: ~2.5 GB → ~200-400 MB (85-90% reduction)
- ⚡ Training speed: ~2x faster
- 📊 Accuracy: Minimal loss (<2-3%)
- 💾 Total disk: ~360 GB → ~30-50 GB for all 144 models

**Note:** These parameters are research-appropriate. For production, you might tune further based on backtesting results.

## Step 9: Train All Models

This will train all 144 models (6 feature windows × 6 target windows × 4 model types).

**Note:** This will take significant time. Models already trained will be skipped.

In [15]:
# Execute Optimized Training Pipeline

print("🚀 Starting Optimized Multi-Class Training Pipeline")
print("="*70)
print(f"📊 Dataset: {len(stock_data):,} records, {stock_data['symbol'].nunique()} symbols")
print(f"📅 Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")
print("="*70)

# Train all models with optimized pipeline
training_log = train_all_models_optimized(stock_data)

print("\n" + "="*70)
print("✅ TRAINING PIPELINE COMPLETE!")
print("="*70)
print(f"📊 Trained {len(training_log)} models")
print(f"💾 Models saved to: {MODELS_DIR}")
print(f"📈 Metrics saved to: {RESULTS_DIR}")
print(f"📝 Training log: {LOG_FILE}")


🚀 Starting Optimized Multi-Class Training Pipeline
📊 Dataset: 669,825 records, 285 symbols
📅 Date range: 2015-01-02 00:00:00 to 2024-12-30 00:00:00

🚀 STARTING OPTIMIZED TRAINING PIPELINE
📊 Total models to train: 180
   Feature windows: [5, 10, 15, 20, 25, 30]
   Target windows: [5, 10, 15, 20, 25, 30]
   Model types: ['RandomForest', 'GradientBoosting', 'XGBoost', 'SVM', 'LSTM']

💾 Feature cache directory: data/research/features_cache

STEP 1: PRE-CALCULATING FEATURES (with caching)

🔧 PRE-CALCULATING FEATURES FOR ALL WINDOWS

📊 Window 5 days:
   ♻️  Loading cached features from: data/research/features_cache/features_5d.pkl
   ✅ Loaded 60 columns, 669,825 records

📊 Window 10 days:
   ♻️  Loading cached features from: data/research/features_cache/features_10d.pkl
   ✅ Loaded 60 columns, 669,825 records

📊 Window 15 days:
   ♻️  Loading cached features from: data/research/features_cache/features_15d.pkl
   ✅ Loaded 60 columns, 669,825 records

📊 Window 20 days:
   ♻️  Loading cached fe

2025-10-23 14:40:13.024476: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 419573160 exceeds 10% of free system memory.
2025-10-23 14:40:49.077477: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 46620400 exceeds 10% of free system memory.


        ✅ LSTM trained in 6 epochs
           Train accuracy: 0.3871, Val accuracy: 0.3865


2025-10-23 14:43:34.227744: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 310794160 exceeds 10% of free system memory.


8373/8373 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
     ✅ LSTM: accuracy=0.386, f1=0.216

  📊 Progress: 20/180 (11.1%)
     ✅ Trained: 7, ⏭️  Skipped: 13

  🎯 TARGET WINDOW: 25 days
     🔧 Excluded columns: ['symbol', 'date']
     📊 Training data: 669,825 samples, 58 features

     🔧 Training RandomForest...
     🔍 RandomForest - Time-based split:
        Train: 401,895 samples (earliest 60.0%)
        Test:  267,930 samples (latest 40.0%)
        Train classes: {0: 136993, 1: 154905, 2: 109997}
        Test classes:  {0: 91480, 1: 103077, 2: 73373}
        ✅ Clean data: 58 features
     ✅ RandomForest: accuracy=0.545, f1=0.546

     🔧 Training GradientBoosting...
     🔍 GradientBoosting - Time-based split:
        Train: 401,895 samples (earliest 60.0%)
        Test:  267,930 samples (latest 40.0%)
        Train classes: {0: 136993, 1: 154905, 2: 109997}
        Test classes:  {0: 91480, 1: 103077, 2: 73373}
        ✅ Clean data: 58 features
     ✅ GradientBoosting: accuracy=0.626, f1=0.608



2025-10-23 15:09:14.242208: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 419573160 exceeds 10% of free system memory.
2025-10-23 15:09:50.454167: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 46620400 exceeds 10% of free system memory.


        ✅ LSTM trained in 6 epochs
           Train accuracy: 0.3857, Val accuracy: 0.3840
8373/8373 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step
     ✅ LSTM: accuracy=0.385, f1=0.215

  📊 Progress: 25/180 (13.9%)
     ✅ Trained: 12, ⏭️  Skipped: 13

  🎯 TARGET WINDOW: 30 days
     🔧 Excluded columns: ['symbol', 'date']
     📊 Training data: 669,825 samples, 58 features

     🔧 Training RandomForest...
     🔍 RandomForest - Time-based split:
        Train: 401,895 samples (earliest 60.0%)
        Test:  267,930 samples (latest 40.0%)
        Train classes: {0: 134115, 1: 162104, 2: 105676}
        Test classes:  {0: 89656, 1: 107830, 2: 70444}
        ✅ Clean data: 58 features
     ✅ RandomForest: accuracy=0.526, f1=0.526

     🔧 Training GradientBoosting...
     🔍 GradientBoosting - Time-based split:
        Train: 401,895 samples (earliest 60.0%)
        Test:  267,930 samples (latest 40.0%)
        Train classes: {0: 134115, 1: 162104, 2: 105676}
        Test classes:  {0: 89656, 1: 107830, 2: 

## Step 10: View Training Progress

## Step 11: Load Trained Models

In [16]:
def load_trained_model(feature_window, target_window, model_name):
    """
    Load a trained model
    
    Returns:
    --------
    model : trained model
    scaler : fitted scaler
    metrics : dict with evaluation metrics
    """
    model_file = get_model_filename(feature_window, target_window, model_name)
    metrics_file = get_metrics_filename(feature_window, target_window, model_name)
    
    # Load model
    if model_name == 'LSTM':
        model = keras.models.load_model(f"{model_file}.h5")
        with open(f"{model_file}_scaler.pkl", 'rb') as f:
            scaler = pickle.load(f)
    else:
        with open(f"{model_file}.pkl", 'rb') as f:
            data = pickle.load(f)
            model = data['model']
            scaler = data['scaler']
    
    # Load metrics
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)
    
    return model, scaler, metrics

# Example: Load a model
# model, scaler, metrics = load_trained_model(5, 5, 'RandomForest')
# print(f"Loaded model with accuracy: {metrics['accuracy']:.3f}")

print("✅ Model loading function ready!")

✅ Model loading function ready!


## Step 12: Model Performance Analysis

In [17]:
def analyze_model_performance():
    """
    Analyze performance of all trained models
    """
    log = load_training_log()
    
    if not log:
        print("❌ No trained models found")
        return
    
    print("\n📊 MODEL PERFORMANCE ANALYSIS")
    print("="*70)
    
    # Collect all metrics
    results = []
    for key, value in log.items():
        if value.get('status') != 'completed':
            continue
        
        metrics = value.get('metrics', {})
        results.append({
            'model': key,
            'feature_window': value['feature_window'],
            'target_window': value['target_window'],
            'model_name': value['model_name'],
            'accuracy': metrics.get('accuracy', 0),
            'f1_macro': metrics.get('f1_macro', 0)
        })
    
    if not results:
        print("❌ No completed models found")
        return
    
    df_results = pd.DataFrame(results)
    
    # Overall statistics
    print(f"\n📈 Overall Statistics:")
    print(f"   Total models: {len(df_results)}")
    print(f"   Average accuracy: {df_results['accuracy'].mean():.3f}")
    print(f"   Average F1 score: {df_results['f1_macro'].mean():.3f}")
    print(f"   Best accuracy: {df_results['accuracy'].max():.3f}")
    print(f"   Worst accuracy: {df_results['accuracy'].min():.3f}")
    
    # Best models
    print(f"\n🏆 Top 10 Models by Accuracy:")
    top_models = df_results.nlargest(10, 'accuracy')
    for idx, row in top_models.iterrows():
        print(f"   {row['model']}: Acc={row['accuracy']:.3f}, F1={row['f1_macro']:.3f}")
    
    # Performance by model type
    print(f"\n📊 Average Performance by Model Type:")
    by_model = df_results.groupby('model_name')[['accuracy', 'f1_macro']].mean()
    for model_name, row in by_model.iterrows():
        print(f"   {model_name}: Acc={row['accuracy']:.3f}, F1={row['f1_macro']:.3f}")
    
    # Performance by feature window
    print(f"\n📊 Average Performance by Feature Window:")
    by_feature = df_results.groupby('feature_window')[['accuracy', 'f1_macro']].mean()
    for fw, row in by_feature.iterrows():
        print(f"   {fw}-day: Acc={row['accuracy']:.3f}, F1={row['f1_macro']:.3f}")
    
    # Performance by target window
    print(f"\n📊 Average Performance by Target Window:")
    by_target = df_results.groupby('target_window')[['accuracy', 'f1_macro']].mean()
    for tw, row in by_target.iterrows():
        print(f"   {tw}-day: Acc={row['accuracy']:.3f}, F1={row['f1_macro']:.3f}")
    
    # Create visualization
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Distribution of accuracy
    axes[0, 0].hist(df_results['accuracy'], bins=20, edgecolor='black')
    axes[0, 0].set_title('Distribution of Model Accuracy')
    axes[0, 0].set_xlabel('Accuracy')
    axes[0, 0].set_ylabel('Count')
    
    # 2. Performance by model type
    by_model[['accuracy', 'f1_macro']].plot(kind='bar', ax=axes[0, 1])
    axes[0, 1].set_title('Performance by Model Type')
    axes[0, 1].set_ylabel('Score')
    axes[0, 1].legend(['Accuracy', 'F1 Score'])
    axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45)
    
    # 3. Performance by feature window
    by_feature[['accuracy', 'f1_macro']].plot(kind='line', marker='o', ax=axes[1, 0])
    axes[1, 0].set_title('Performance by Feature Window')
    axes[1, 0].set_xlabel('Feature Window (days)')
    axes[1, 0].set_ylabel('Score')
    axes[1, 0].legend(['Accuracy', 'F1 Score'])
    axes[1, 0].grid(True)
    
    # 4. Performance by target window
    by_target[['accuracy', 'f1_macro']].plot(kind='line', marker='o', ax=axes[1, 1])
    axes[1, 1].set_title('Performance by Target Window')
    axes[1, 1].set_xlabel('Target Window (days)')
    axes[1, 1].set_ylabel('Score')
    axes[1, 1].legend(['Accuracy', 'F1 Score'])
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/performance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n💾 Visualization saved to: {RESULTS_DIR}/performance_analysis.png")
    
    return df_results

# Analyze performance (uncomment to run)
# results_df = analyze_model_performance()

print("✅ Performance analysis function ready!")

✅ Performance analysis function ready!


## 🎉 Next Steps

1. **Train Models**: Uncomment the training code in Step 9 to train all 144 models
2. **Monitor Progress**: Use Step 10 to view training progress  
3. **Analyze Performance**: Use Step 12 to analyze model performance
4. **Implement Backtesting**: Use trained models for backtesting (see separate notebook)

**Note:** Training all 144 models will take several hours. The system will automatically skip models that are already trained, so you can run this notebook multiple times safely.

## 📊 Training Overview

- **Total Models:** 144 (6 feature windows × 6 target windows × 4 model types)
- **Feature Windows:** 5, 10, 15, 20, 25, 30 days
- **Target Windows:** 5, 10, 15, 20, 25, 30 days
- **Models:** RandomForest, GradientBoosting, SVM, LSTM
- **Classes:** Momentum (0), Reversal (1), Do Nothing (2)

## 💡 Tips

- Start with a few models first to test the system
- Monitor memory usage (LSTM models use more memory)
- Use the persistence system - already trained models will be skipped
- Check training log regularly with `view_training_progress()`
- Analyze results with `analyze_model_performance()` after training

In [19]:
#!/usr/bin/env python3
"""
Quick script to check feature importance and identify fundamental features
"""
import pandas as pd
import pickle
import json
import os
from collections import defaultdict

DATA_DIR = 'data/research'
MODELS_DIR = os.path.join(DATA_DIR, 'multiclass_models')
LOG_FILE = os.path.join(DATA_DIR, 'training_log.json')
FEATURES_CACHE_DIR = os.path.join(DATA_DIR, 'features_cache')

# Load training log
with open(LOG_FILE, 'r') as f:
    training_log = json.load(f)

# Fundamental feature columns (from the data)
FUNDAMENTAL_FEATURES = [
    'pe_ratio', 'forward_pe', 'price_to_book', 'price_to_sales', 'market_cap',
    'enterprise_value', 'debt_to_equity', 'return_on_equity', 'return_on_assets',
    'profit_margins', 'operating_margin', 'gross_margin', 'revenue_growth',
    'earnings_growth', 'dividend_yield', 'payout_ratio', 'beta',
    'free_cashflow', 'operating_cashflow', 'total_revenue', 'net_income',
    'eps', 'book_value', 'ev_to_revenue', 'ev_to_ebitda', 'quick_ratio',
    'current_ratio', 'revenue_growth_yearly', 'earnings_growth_yearly',
    'dividend_rate', 'shares_outstanding', 'float_shares'
]

print("🔍 CHECKING FEATURE IMPORTANCE")
print("="*80)

# Sample a few RandomForest and GradientBoosting models (they have feature_importances_)
sample_models = [
    'f5d_t5d_RandomForest',
    'f10d_t10d_RandomForest',
    'f5d_t5d_GradientBoosting',
    'f10d_t10d_GradientBoosting',
    'f5d_t5d_XGBoost'
]

all_importances = defaultdict(list)

for model_key in sample_models:
    model_path = os.path.join(MODELS_DIR, f'{model_key}.pkl')
    
    if not os.path.exists(model_path):
        print(f"⚠️  {model_key} not found")
        continue
    
    print(f"\n📊 Loading {model_key}...")
    
    try:
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        
        # Get feature importance
        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            
            # Load corresponding features to get names
            feature_window = int(model_key.split('_')[0][1:-1])
            cache_file = os.path.join(FEATURES_CACHE_DIR, f'features_{feature_window}d.pkl')
            
            if os.path.exists(cache_file):
                features_df = pd.read_pickle(cache_file)
                feature_names = [col for col in features_df.columns 
                               if col not in ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']]
                
                # Create importance dict
                importance_dict = dict(zip(feature_names, importances))
                
                # Separate fundamental vs other features
                fundamental_importance = 0
                other_importance = 0
                
                for feat_name, importance in importance_dict.items():
                    is_fundamental = any(fund in feat_name for fund in FUNDAMENTAL_FEATURES)
                    if is_fundamental:
                        fundamental_importance += importance
                        all_importances[feat_name].append(importance)
                    else:
                        other_importance += importance
                
                total = fundamental_importance + other_importance
                fund_pct = (fundamental_importance / total * 100) if total > 0 else 0
                
                print(f"   Fundamental features: {fund_pct:.1f}%")
                print(f"   Other features: {100-fund_pct:.1f}%")
                
                # Show top 5 features
                top_5 = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:5]
                print(f"\n   Top 5 features:")
                for feat, imp in top_5:
                    is_fund = any(fund in feat for fund in FUNDAMENTAL_FEATURES)
                    marker = "💰" if is_fund else "📊"
                    print(f"      {marker} {feat}: {imp:.4f}")
        
        else:
            print(f"   ⚠️  Model doesn't have feature_importances_")
    
    except Exception as e:
        print(f"   ❌ Error: {e}")

# Summary
print("\n" + "="*80)
print("📊 SUMMARY: FUNDAMENTAL FEATURES IMPORTANCE")
print("="*80)

if all_importances:
    # Get top fundamental features across all models
    avg_importance = {feat: sum(imps)/len(imps) for feat, imps in all_importances.items()}
    top_fundamentals = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)[:10]
    
    print("\nTop 10 Fundamental Features (average importance):")
    for feat, imp in top_fundamentals:
        print(f"   {feat}: {imp:.4f}")
    
    total_fund_importance = sum(avg_importance.values())
    print(f"\nTotal fundamental importance: {total_fund_importance:.4f}")
    
    if total_fund_importance < 0.1:
        print("\n✅ VERDICT: Fundamental features have LOW importance (<10%)")
        print("   → Safe to keep current data with static fundamentals")
        print("   → Document as limitation in paper")
    elif total_fund_importance < 0.25:
        print("\n⚠️  VERDICT: Fundamental features have MEDIUM importance (10-25%)")
        print("   → Consider retraining without fundamentals")
        print("   → Or fix the time-varying issue")
    else:
        print("\n❌ VERDICT: Fundamental features have HIGH importance (>25%)")
        print("   → Need to fix! Either:")
        print("      1. Remove fundamentals and retrain")
        print("      2. Implement proper time-varying fundamentals")
else:
    print("❌ Could not extract feature importance")



🔍 CHECKING FEATURE IMPORTANCE

📊 Loading f5d_t5d_RandomForest...
   Fundamental features: 30.0%
   Other features: 70.0%

   Top 5 features:
      💰 mom_market_cap_std: 0.1469
      💰 rev_market_cap_std: 0.1426
      📊 rev_pb: 0.1328
      📊 macd_histogram: 0.1324
      📊 cci: 0.1271

📊 Loading f10d_t10d_RandomForest...
   Fundamental features: 26.3%
   Other features: 73.7%

   Top 5 features:
      📊 rev_pb: 0.1379
      💰 mom_market_cap_std: 0.1304
      📊 cci: 0.1266
      📊 macd_histogram: 0.1252
      📊 mom_pb: 0.1240

📊 Loading f5d_t5d_GradientBoosting...
   Fundamental features: 34.4%
   Other features: 65.6%

   Top 5 features:
      💰 rev_market_cap_std: 0.1725
      💰 mom_market_cap_std: 0.1705
      📊 rev_pb: 0.1680
      📊 macd_histogram: 0.1607
      📊 mom_pb: 0.1584

📊 Loading f10d_t10d_GradientBoosting...
   Fundamental features: 31.7%
   Other features: 68.3%

   Top 5 features:
      💰 rev_market_cap_std: 0.1653
      📊 rev_pb: 0.1634
      📊 cci: 0.1605
      📊 mom_p